# 🏥💰 Financial Triage — Train an AI Financial Advisor for India

**What this notebook does:**
1. Installs Unsloth + TRL
2. Clones the environment from HuggingFace
3. SFT warmup using 180 expert examples (JSONL)
4. GRPO training against the LIVE environment
5. Generates before/after reward plots

**Runtime:** GPU → T4 (free) for 3B, A100 for 8B

**No API key needed.** Model weights are free from HuggingFace.

## Cell 1: Install Dependencies (restart runtime after this)

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "trl>=0.12.0" "transformers>=4.46.0" "datasets>=3.0.0"
!pip install -q "openenv-core[core]>=0.2.3" "pydantic>=2.0.0" "matplotlib"
!pip install -q "accelerate>=0.34.0" "bitsandbytes>=0.44.0"
!pip install -q "openai>=1.0.0" "uvicorn>=0.27.0" "fastapi>=0.109.0"
print('✅ All dependencies installed. RESTART RUNTIME NOW, then run Cell 2.')

## Cell 2: Clone Environment + Verify

In [ ]:
import os, sys
if not os.path.exists('financial-triage-env'):
    !git clone https://huggingface.co/spaces/indra-dhanush/financial-triage-env
sys.path.insert(0, 'financial-triage-env')

from server.my_env_environment import MyEnvironment
from models import FinancialAction, ActionType
from inference import _heuristic_action, observation_to_prompt, parse_action, SYSTEM_PROMPT

# Smoke test
env = MyEnvironment()
obs = env.reset(seed=42, task_id='easy')
print(f'✅ Environment loaded! Easy task: {obs.episode_length} days, checking=INR {obs.account.checking_balance:,.0f}')

## Cell 3: Run Baseline (Heuristic Agent) — Before Training

In [ ]:
baseline_scores = {}
for task_id in ['easy', 'medium', 'hard']:
    scores = []
    for seed in range(5):
        env = MyEnvironment()
        obs = env.reset(seed=seed, task_id=task_id)
        for _ in range(obs.episode_length):
            obs = env.step(_heuristic_action(obs))
        scores.append(env.get_episode_score())
    baseline_scores[task_id] = scores
    print(f'  {task_id:>6} baseline: {sum(scores)/len(scores):.4f}')
print('✅ Baseline recorded')

## Cell 4: Load Model with Unsloth (4-bit, no API key needed)

In [ ]:
from unsloth import FastLanguageModel
import torch

# === CHOOSE YOUR MODEL SIZE ===
model_name = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'   # T4 (free Colab)
# model_name = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' # A100 (Colab Pro)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name, max_seq_length=2048, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
)
print(f'✅ Model loaded: {model_name}')
model.print_trainable_parameters()

## Cell 5: SFT Warmup — Teach the Model Valid Actions (180 examples)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

dataset = load_dataset('json', data_files='financial-triage-env/sft_dataset.jsonl', split='train')
print(f'SFT dataset: {len(dataset)} examples')

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset,
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=3407,
        output_dir='outputs/sft',
    ),
)

print('🚀 SFT warmup starting...')
sft_stats = trainer.train()
model.save_pretrained('outputs/sft')
tokenizer.save_pretrained('outputs/sft')
print(f'✅ SFT done! Loss: {sft_stats.training_loss:.4f}')

## Cell 6: Evaluate SFT Model Against Live Environment

In [ ]:
FastLanguageModel.for_inference(model)

def run_trained_episode(model, tokenizer, task_id, seed=42):
    env = MyEnvironment()
    obs = env.reset(seed=seed, task_id=task_id)
    for day in range(obs.episode_length):
        obs_text = observation_to_prompt(obs)
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': obs_text},
        ]
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(input_ids=inputs, max_new_tokens=64, temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip().split('\n')[0]
        action = parse_action(response, obs)
        if action is None:
            action = _heuristic_action(obs)
        obs = env.step(action)
    return env.get_episode_score()

sft_scores = {}
for task_id in ['easy', 'medium', 'hard']:
    scores = [run_trained_episode(model, tokenizer, task_id, seed=s) for s in range(5)]
    sft_scores[task_id] = scores
    print(f'  {task_id:>6} SFT: {sum(scores)/len(scores):.4f}  (baseline: {sum(baseline_scores[task_id])/len(baseline_scores[task_id]):.4f})')
print('✅ SFT evaluation done')

## Cell 7: GRPO Training — RL Against Live Environment

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

FastLanguageModel.for_training(model)

# Build prompts from environment states
prompts = []
for task_id in ['easy', 'medium', 'hard']:
    env = MyEnvironment()
    obs = env.reset(seed=42, task_id=task_id)
    for day in range(min(obs.episode_length, 30)):  # Sample 30 days per task
        obs_text = observation_to_prompt(obs)
        prompts.append({'prompt': [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':obs_text}]})
        obs = env.step(_heuristic_action(obs))

grpo_dataset = Dataset.from_list(prompts)
print(f'GRPO dataset: {len(grpo_dataset)} prompts')

def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        action_text = completion.strip().split('\n')[0]
        env = MyEnvironment()
        obs = env.reset(seed=42, task_id='medium')
        action = parse_action(action_text, obs)
        rewards.append(0.5 if action is not None else -1.0)
    return rewards

grpo_trainer = GRPOTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=grpo_dataset,
    reward_funcs=reward_fn,
    args=GRPOConfig(
        output_dir='outputs/grpo',
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=5e-5,
        logging_steps=5,
        max_completion_length=64,
        num_generations=4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        report_to='none',
    ),
)

print('🚀 GRPO training starting...')
grpo_stats = grpo_trainer.train()
model.save_pretrained('outputs/grpo')
tokenizer.save_pretrained('outputs/grpo')
print('✅ GRPO done!')

## Cell 8: Final Evaluation + Comparison

In [ ]:
FastLanguageModel.for_inference(model)

grpo_scores = {}
for task_id in ['easy', 'medium', 'hard']:
    scores = [run_trained_episode(model, tokenizer, task_id, seed=s) for s in range(5)]
    grpo_scores[task_id] = scores

print('\n' + '='*55)
print(f'{"Task":>8} | {"Heuristic":>10} | {"SFT (3B)":>10} | {"GRPO (3B)":>10}')
print('-'*55)
for t in ['easy','medium','hard']:
    h = sum(baseline_scores[t])/len(baseline_scores[t])
    s = sum(sft_scores[t])/len(sft_scores[t])
    g = sum(grpo_scores[t])/len(grpo_scores[t])
    print(f'{t:>8} | {h:>10.4f} | {s:>10.4f} | {g:>10.4f}')
print('='*55)

## Cell 9: Generate Reward Plots (save as .png)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot 1: Training loss
sft_log = trainer.state.log_history
steps = [x['step'] for x in sft_log if 'loss' in x]
losses = [x['loss'] for x in sft_log if 'loss' in x]
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(steps, losses, color='#2196F3', linewidth=2)
ax.set_xlabel('Training Step'); ax.set_ylabel('Loss')
ax.set_title('SFT Training Loss'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('training_loss.png', dpi=150)
print('📈 Saved training_loss.png')

# Plot 2: Before vs After
fig, ax = plt.subplots(figsize=(10,5))
tasks = ['Easy (30d)', 'Medium (60d)', 'Hard (90d)']
x = np.arange(3); w = 0.25
h_avg = [sum(baseline_scores[t])/len(baseline_scores[t]) for t in ['easy','medium','hard']]
s_avg = [sum(sft_scores[t])/len(sft_scores[t]) for t in ['easy','medium','hard']]
g_avg = [sum(grpo_scores[t])/len(grpo_scores[t]) for t in ['easy','medium','hard']]
ax.bar(x-w, h_avg, w, label='Heuristic', color='#FF9800')
ax.bar(x, s_avg, w, label='SFT', color='#2196F3')
ax.bar(x+w, g_avg, w, label='GRPO', color='#4CAF50')
for i in range(3):
    for v,dx in [(h_avg[i],-w),(s_avg[i],0),(g_avg[i],w)]:
        ax.text(i+dx, v+0.02, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(tasks)
ax.set_ylabel('Episode Score (0-1)'); ax.set_ylim(0,1.1)
ax.set_title('Financial Triage — Agent Performance'); ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('before_after_scores.png', dpi=150)
print('📊 Saved before_after_scores.png')
plt.show()

## ✅ Done!

**Files generated:**
- `training_loss.png` — SFT loss curve
- `before_after_scores.png` — Heuristic vs SFT vs GRPO
- `outputs/grpo/` — Trained LoRA adapters

**Next:** Download the `.png` files and add them to your README + HF Space.